# Ball Tracker + Player-Ball Carrier Assigner + Annotation

v3: adds Phase 2 (proximity fallback in pitch space) to the carrier assigner, and
a non-sticky "clear on no candidate" rule to stop stale carriers persisting when
the ball is in transit near nobody. Ball tracker (RTS backward smoothing) and
carrier trust-gating (`decision_sources` / `display_sources`) are unchanged from v2.

Builds on your existing caches:
- `detection_cache` (per-frame detections, `source=="ball_model"` for the ball)
- `tracking_cache` / `locked_class_by_id` (player/gk/ref tracks with global IDs)
- `homography_cache` (per-frame H matrices, `PX_PER_METER=10`)

## 1. Setup + load caches

In [1]:
import sys
sys.path.append('.')
import paths

import pickle
import numpy as np
from collections import defaultdict, deque

with open(paths.DETECTION_CACHE_PATH, 'rb') as f:
    detection_cache = pickle.load(f)

with open(paths.TRACKING_CACHE_PATH, 'rb') as f:
    _tr = pickle.load(f)
    tracking_cache = _tr['tracking_cache']
    locked_class_by_id = _tr['locked_class_by_id']

with open(paths.HOMOGRAPHY_CACHE_PATH, 'rb') as f:
    homography_cache = pickle.load(f)  # list[np.ndarray | None], indexed by frame

PX_PER_METER = 10.0

print("Frames in detection_cache:", len(detection_cache))
print("Frames in tracking_cache:", len(tracking_cache))
print("Frames in homography_cache:", len(homography_cache))

Frames in detection_cache: 70326
Frames in tracking_cache: 70326
Frames in homography_cache: 70327


## 2. Config

In [2]:
class BallTrackerConfig:
    base_process_noise_pos = 15.0
    base_process_noise_vel = 40.0
    measurement_noise = 25.0

    gap_inflation_per_frame = 1.15
    max_gap_inflation = 25.0

    mahalanobis_gate = 9.21

    min_detection_conf = 0.25          # your actual conf threshold
    low_conf_flag = 0.35

    max_track_gap_frames = 38          # 1.5s at 25fps (this clip's actual frame rate)
    min_frames_to_confirm_track = 2

    px_per_meter = 10.0
    fps = 25.0

    # Physical sanity gate, independent of the Mahalanobis/covariance gate.
    # After a gap, P is inflated on purpose (so the filter can re-acquire the ball),
    # which also makes the Mahalanobis gate loose enough for a false-positive
    # detection to sneak through and get accepted for one frame before the next
    # real detection (or the RTS pass) pulls the track back. This caps accepted
    # detections to a physically plausible ball speed regardless of how loose P is.
    max_ball_speed_mps = 35.0          # ~126 km/h, generous ceiling for real shots/clearances

    # NEW -- crowd-aware confidence floor. A ball detection near min_detection_conf
    # is far more likely to actually be a boot/shin-guard false positive when it
    # sits inside a duel/scramble (multiple players clustered near the ball) than
    # when it's out in open space. This raises the bar specifically in crowded
    # situations instead of raising min_detection_conf globally, which would also
    # throw away legitimate low-confidence detections in open play (e.g. partial
    # occlusion, motion blur, far-side-of-pitch detections).
    crowd_min_conf = 0.50              # required confidence when the ball is in a crowd
    crowd_radius_px = 60.0             # "near the ball" radius, in image pixels
    crowd_min_players = 2              # how many players nearby counts as "a crowd"

cfg = BallTrackerConfig()


## 3. Kalman filter -- constant-velocity model with gap-adaptive gating

State: `[x, y, vx, vy]` in pixel space. Forward pass only -- section 6 wraps this
with a backward pass and RTS smoothing.

In [3]:
class BallKalmanTracker:
    def __init__(self, cfg: BallTrackerConfig):
        self.cfg = cfg
        self.x = None
        self.P = None
        self.gap_frames = 0
        self.initialized = False

        self.F = np.array([
            [1, 0, 1, 0],
            [0, 1, 0, 1],
            [0, 0, 1, 0],
            [0, 0, 0, 1],
        ], dtype=float)

        self.H = np.array([
            [1, 0, 0, 0],
            [0, 1, 0, 0],
        ], dtype=float)

    def _process_noise_Q(self):
        q_pos = self.cfg.base_process_noise_pos
        q_vel = self.cfg.base_process_noise_vel
        return np.diag([q_pos, q_pos, q_vel, q_vel])

    def _gap_inflation(self):
        factor = self.cfg.gap_inflation_per_frame ** self.gap_frames
        return min(factor, self.cfg.max_gap_inflation)

    def init(self, x, y):
        self.x = np.array([x, y, 0.0, 0.0])
        self.P = np.diag([self.cfg.measurement_noise, self.cfg.measurement_noise, 100.0, 100.0])
        self.gap_frames = 0
        self.initialized = True

    def predict(self):
        if not self.initialized:
            return None, None
        self.x = self.F @ self.x
        inflation = self._gap_inflation()
        self.P = self.F @ self.P @ self.F.T + self._process_noise_Q() * inflation
        return self.x[:2].copy(), self.P[:2, :2].copy()

    def mahalanobis(self, meas_xy):
        pred_xy = self.x[:2]
        S = self.P[:2, :2] + np.eye(2) * self.cfg.measurement_noise
        diff = np.array(meas_xy) - pred_xy
        try:
            S_inv = np.linalg.inv(S)
        except np.linalg.LinAlgError:
            return np.inf
        return float(diff @ S_inv @ diff.T)

    def update(self, meas_xy):
        R = np.eye(2) * self.cfg.measurement_noise
        S = self.H @ self.P @ self.H.T + R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        y = np.array(meas_xy) - self.H @ self.x
        self.x = self.x + K @ y
        self.P = (np.eye(4) - K @ self.H) @ self.P
        self.gap_frames = 0

    def mark_missed(self):
        self.gap_frames += 1

## 4. Per-frame ball extraction from detection_cache

Pulls the ball entry from your existing detection cache format
(`{bbox, conf, class, source, low_confidence}` per frame, `class == "ball"`).

Also applies a crowd-aware confidence floor: a marginal-confidence detection
close to a cluster of players (a duel/scramble) needs `cfg.crowd_min_conf`,
not just `cfg.min_detection_conf` -- boot/shin-guard false positives cluster
specifically in these crowded frames.

In [4]:
def _count_nearby_players(pred_xy_px, frame_tracks, locked_class_by_id, radius_px):
    """How many player/goalkeeper tracks have their bbox center within radius_px
    of the predicted ball position -- used to detect a duel/scramble."""
    if pred_xy_px is None:
        return 0
    px, py = pred_xy_px
    count = 0
    for t in frame_tracks:
        if locked_class_by_id.get(t['track_id']) not in ('player', 'goalkeeper'):
            continue
        x0, y0, x1, y1 = t['bbox']
        cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
        if (cx - px) ** 2 + (cy - py) ** 2 <= radius_px ** 2:
            count += 1
    return count


def get_ball_detection(detection_cache, frame_idx, min_conf, cfg=None,
                        tracking_cache=None, locked_class_by_id=None, pred_xy_px=None):
    """NEW -- crowd-aware confidence floor. When cfg + tracking_cache +
    locked_class_by_id + pred_xy_px are all supplied, a candidate near a cluster
    of players (a duel/scramble -- where boot/shin-guard false positives cluster)
    needs to clear cfg.crowd_min_conf instead of just min_conf. Falls back to the
    plain min_conf filter (original behavior) when the extra context isn't given,
    so this stays usable standalone (e.g. for quick debug lookups)."""
    dets = detection_cache.get(frame_idx, [])
    ball_dets = [d for d in dets if d.get('class') == 'ball' and d.get('conf', 0) >= min_conf]
    if not ball_dets:
        return None
    best = max(ball_dets, key=lambda d: d['conf'])

    if cfg is not None and tracking_cache is not None and locked_class_by_id is not None:
        frame_tracks = tracking_cache.get(frame_idx, {}).get('tracks', [])
        n_nearby = _count_nearby_players(pred_xy_px, frame_tracks, locked_class_by_id, cfg.crowd_radius_px)
        if n_nearby >= cfg.crowd_min_players and best['conf'] < cfg.crowd_min_conf:
            return None

    bbox = best['bbox']
    cx = (bbox[0] + bbox[2]) / 2
    cy = (bbox[1] + bbox[3]) / 2
    return {'xy': (cx, cy), 'conf': best['conf']}

## 5. Homography helper -- pixel to pitch coordinates

In [5]:
def px_to_pitch(xy_px, H):
    if H is None:
        return None
    pt = np.array([xy_px[0], xy_px[1], 1.0])
    proj = H @ pt
    if abs(proj[2]) < 1e-8:
        return None
    x_m = proj[0] / proj[2] / PX_PER_METER
    y_m = proj[1] / proj[2] / PX_PER_METER
    return (x_m, y_m)

## 6. RTS backward smoother

Run backward from the detection that ends a gap, blended with the forward pass,
so gap frames aren't purely forward-extrapolated (which is confidently wrong
whenever the ball changes direction mid-gap).

In [6]:
def rts_smooth(F, x_filt, P_filt, x_pred, P_pred):
    """RTS backward smoother over one continuous track segment."""
    n = len(x_filt)
    x_smooth = [None] * n
    P_smooth = [None] * n
    x_smooth[-1] = x_filt[-1]
    P_smooth[-1] = P_filt[-1]

    for k in range(n - 2, -1, -1):
        try:
            P_pred_inv = np.linalg.inv(P_pred[k + 1])
        except np.linalg.LinAlgError:
            P_pred_inv = np.linalg.pinv(P_pred[k + 1])
        C = P_filt[k] @ F.T @ P_pred_inv
        x_smooth[k] = x_filt[k] + C @ (x_smooth[k + 1] - x_pred[k + 1])
        P_smooth[k] = P_filt[k] + C @ (P_smooth[k + 1] - P_pred[k + 1]) @ C.T

    return x_smooth, P_smooth

## 7. Run the tracker over the full match

Output schema per frame:
```
{frame_idx: {
    "xy_px": (x, y) or None,
    "xy_pitch": (x, y) or None,
    "source": "detected" | "smoothed" | "lost",
    "conf": float or None,
}}
```
`detected`: real detection passed the gate. `smoothed`: no detection, but a
future detection exists later in the same track segment (backward-informed).
`lost`: no detection and no future anchor to confirm a position (tail of a
still-open segment, or gap exceeded `max_track_gap_frames`) -- the 'predicted'
(pure forward-extrapolation) pipeline was removed since it wasn't trustworthy
enough to use; these frames are now just unknown (`xy_px=None`).

In [7]:
def run_ball_tracker(detection_cache, tracking_cache, locked_class_by_id,
                      homography_cache, cfg: BallTrackerConfig, total_frames):
    tracker = BallKalmanTracker(cfg)
    ball_tracked_cache = {}
    stats = defaultdict(int)

    seg_frames = []
    seg_x_filt = []
    seg_P_filt = []
    seg_x_pred = []
    seg_P_pred = []
    seg_was_detected = []
    seg_raw_meas = []
    seg_conf = []

    def flush_segment():
        if not seg_frames:
            return
        x_smooth, P_smooth = rts_smooth(tracker.F, seg_x_filt, seg_P_filt, seg_x_pred, seg_P_pred)

        detected_positions = [i for i, d in enumerate(seg_was_detected) if d]
        last_detected_idx = max(detected_positions) if detected_positions else -1

        for i, f in enumerate(seg_frames):
            H = homography_cache[f] if f < len(homography_cache) else None
            if seg_was_detected[i]:
                xy_px = seg_raw_meas[i]
                source = 'detected'
                conf = seg_conf[i]
            elif i < last_detected_idx:
                xy_px = tuple(x_smooth[i][:2])
                source = 'smoothed'
                conf = None
            else:
                # NEW -- removed the 'predicted' pipeline. A frame after the last
                # real detection in a still-open segment has no closing detection
                # to confirm it, so pure forward extrapolation isn't trustworthy
                # (the ball can bounce/deflect anywhere). Treat it as unknown
                # instead of drawing/using a guessed position.
                xy_px = None
                source = 'lost'
                conf = None

            ball_tracked_cache[f] = {
                'xy_px': xy_px,
                'xy_pitch': px_to_pitch(xy_px, H) if xy_px is not None else None,
                'source': source, 'conf': conf,
            }
            stats[source] += 1

        seg_frames.clear(); seg_x_filt.clear(); seg_P_filt.clear()
        seg_x_pred.clear(); seg_P_pred.clear()
        seg_was_detected.clear(); seg_raw_meas.clear(); seg_conf.clear()

    def _speed_ok(det_xy, last_xy_px, gap_frames, frame_idx):
        """NEW -- reject detections that imply an unrealistic ball speed, on top of
        the Mahalanobis gate. Only checked after a miss, since adjacent-frame jumps
        are already tightly constrained by the gate."""
        if gap_frames == 0:
            return True
        H_now = homography_cache[frame_idx] if frame_idx < len(homography_cache) else None
        cur_pitch = px_to_pitch(det_xy, H_now)
        last_pitch = px_to_pitch(tuple(last_xy_px), H_now)
        if cur_pitch is None or last_pitch is None:
            return True  # can't evaluate without homography -- fall back to Mahalanobis only
        elapsed_s = (gap_frames + 1) / cfg.fps
        implied_speed = (((cur_pitch[0] - last_pitch[0]) ** 2 +
                           (cur_pitch[1] - last_pitch[1]) ** 2) ** 0.5) / elapsed_s
        return implied_speed <= cfg.max_ball_speed_mps

    for frame_idx in range(total_frames):
        if not tracker.initialized:
            # NEW -- no predicted position yet, so the crowd check can't run --
            # falls back to the plain min_detection_conf filter for this frame.
            det = get_ball_detection(detection_cache, frame_idx, cfg.min_detection_conf,
                                      cfg=cfg, tracking_cache=tracking_cache,
                                      locked_class_by_id=locked_class_by_id, pred_xy_px=None)
            if det is not None:
                tracker.init(*det['xy'])
                seg_frames.append(frame_idx)
                seg_x_filt.append(tracker.x.copy())
                seg_P_filt.append(tracker.P.copy())
                seg_x_pred.append(None)
                seg_P_pred.append(None)
                seg_was_detected.append(True)
                seg_raw_meas.append(det['xy'])
                seg_conf.append(det['conf'])
            else:
                ball_tracked_cache[frame_idx] = {
                    'xy_px': None, 'xy_pitch': None, 'source': 'lost', 'conf': None,
                }
                stats['lost'] += 1
            continue

        x_pred, P_pred = tracker.predict()
        P_pred_full = tracker.P.copy()
        x_pred_full = tracker.x.copy()

        # NEW -- crowd-aware confidence floor, using the just-computed predicted
        # position to check how many players are clustered around it.
        det = get_ball_detection(detection_cache, frame_idx, cfg.min_detection_conf,
                                  cfg=cfg, tracking_cache=tracking_cache,
                                  locked_class_by_id=locked_class_by_id,
                                  pred_xy_px=tuple(x_pred_full[:2]))

        accepted = False
        if det is not None:
            dist = tracker.mahalanobis(det['xy'])
            gate_ok = dist <= cfg.mahalanobis_gate
            speed_ok = _speed_ok(det['xy'], x_pred_full[:2], tracker.gap_frames, frame_idx) if gate_ok else False
            if gate_ok and speed_ok:
                tracker.update(det['xy'])
                accepted = True
            elif gate_ok and not speed_ok:
                stats['rejected_speed'] += 1

        seg_frames.append(frame_idx)
        seg_x_pred.append(x_pred_full)
        seg_P_pred.append(P_pred_full)

        if accepted:
            seg_x_filt.append(tracker.x.copy())
            seg_P_filt.append(tracker.P.copy())
            seg_was_detected.append(True)
            seg_raw_meas.append(det['xy'])
            seg_conf.append(det['conf'])
        else:
            tracker.mark_missed()
            seg_x_filt.append(x_pred_full)
            seg_P_filt.append(P_pred_full)
            seg_was_detected.append(False)
            seg_raw_meas.append(None)
            seg_conf.append(None)

            if tracker.gap_frames > cfg.max_track_gap_frames:
                tracker.initialized = False
                flush_segment()
                ball_tracked_cache[frame_idx] = {
                    'xy_px': None, 'xy_pitch': None, 'source': 'lost', 'conf': None,
                }
                stats['lost'] += 1

    flush_segment()
    return ball_tracked_cache, dict(stats)

TOTAL_FRAMES = max(detection_cache.keys()) + 1
ball_tracked_cache, stats = run_ball_tracker(
    detection_cache, tracking_cache, locked_class_by_id, homography_cache, cfg, TOTAL_FRAMES
)

print("Frames processed:", TOTAL_FRAMES)
for k, v in stats.items():
    print(f"  {k}: {v} ({100*v/TOTAL_FRAMES:.1f}%)")


Frames processed: 70326
  rejected_speed: 864 (1.2%)
  detected: 38303 (54.5%)
  smoothed: 16035 (22.8%)
  lost: 16153 (23.0%)


## 8. Debug -- gap length distribution

In [8]:
def summarize_gaps(ball_tracked_cache, total_frames):
    gaps = []
    current_gap = 0
    for f in range(total_frames):
        src = ball_tracked_cache[f]['source']
        if src == 'detected':
            if current_gap > 0:
                gaps.append(current_gap)
            current_gap = 0
        else:
            current_gap += 1
    if current_gap > 0:
        gaps.append(current_gap)

    gaps = np.array(gaps)
    if len(gaps) == 0:
        print("No gaps -- every frame detected (unlikely at 50% recall, check input).")
        return

    print(f"Number of gap runs: {len(gaps)}")
    print(f"Mean gap length: {gaps.mean():.1f} frames")
    print(f"Median gap length: {np.median(gaps):.1f} frames")
    print(f"Max gap length: {gaps.max()} frames")
    print(f"Gaps > max_track_gap_frames ({cfg.max_track_gap_frames}): {(gaps > cfg.max_track_gap_frames).sum()}")

    source_counts = defaultdict(int)
    for f in range(total_frames):
        source_counts[ball_tracked_cache[f]['source']] += 1
    print()
    for src, count in source_counts.items():
        print(f"  {src}: {count} ({100*count/total_frames:.1f}%)")

summarize_gaps(ball_tracked_cache, TOTAL_FRAMES)

Number of gap runs: 3116
Mean gap length: 10.3 frames
Median gap length: 3.0 frames
Max gap length: 706 frames
Gaps > max_track_gap_frames (38): 165

  detected: 38303 (54.5%)
  smoothed: 16035 (22.8%)
  lost: 15988 (22.7%)


## 9. Save ball_tracked_cache

In [9]:
import os

os.makedirs(os.path.dirname(paths.BALL_TRACKED_CACHE_PATH), exist_ok=True)
with open(paths.BALL_TRACKED_CACHE_PATH, 'wb') as f:
    pickle.dump(ball_tracked_cache, f)
print(f"Saved to {paths.BALL_TRACKED_CACHE_PATH}")

Saved to barca_atletico_first_half/cache/barca_atletico_first_half_ball_tracked_cache.pkl


---
# Player-Ball Carrier Assigner

- **Phase 1 -- bbox overlap**: ball center within `bbox_overlap_margin_px` of a
  player's bbox -> immediate assignment.
- **Phase 2 -- proximity (NEW, wired in this version)**: no overlap -> fall back
  to the nearest player within `max_carrier_distance_m` in pitch space, using
  each player's bbox bottom-center (feet) projected via the same
  `homography_cache[frame_idx]` / `px_to_pitch()` used for the ball.
- **Hysteresis**: a new candidate must lead for `min_frames_to_switch`
  consecutive frames before possession switches.
- **NEW -- clear_on_no_candidate**: if a decision-eligible frame finds no
  candidate at all (no overlap, nobody within `max_carrier_distance_m`), the
  carrier is cleared immediately rather than sticking to whoever held it last.
  This directly targets the "false carrier persists after the ball has clearly
  moved on" symptom -- previously, `best_candidate=None` just left hysteresis to
  keep the stale value indefinitely.
- Trust gating unchanged from v2: `decision_sources` / `display_sources` =
  `{'detected', 'smoothed'}`, so `predicted` ball frames never drive or display
  a carrier.

## 10. Carrier config

In [10]:
class CarrierConfig:
    bbox_overlap_margin_px = 15.0      # how far outside a player bbox still counts as "touching"
    max_carrier_distance_m = 2.5         # phase 2 fallback radius, in pitch meters
    min_frames_to_switch = 10           # hysteresis -- consecutive frames a new candidate must lead by
    clear_on_no_candidate = True       # don't let hysteresis keep a stale carrier when nobody's near the ball

    no_candidate_grace_frames = 8      # hold the last carrier this many frames (~0.3s @ 25fps)
                                        # before actually clearing, so a loose ball in a duel or a
                                        # single missed-touch frame doesn't flicker the carrier off

    # NEW -- margin-based ambiguity check (handles false positives from 50/50 duels).
    # The best candidate's distance must beat the second-best by this ratio to be
    # trusted; otherwise the frame is "contested" and falls through to the grace-hold
    # path instead of guessing.
    candidate_margin_ratio = 0.7

    # NEW -- velocity-consistency check (handles false positives from through-balls
    # grazing a player's proximity zone without an actual touch). Looks at the ball's
    # pitch-space velocity `velocity_check_window` frames before/after a candidate
    # frame; a real touch usually changes speed and/or direction, a ball merely
    # passing by usually doesn't. Only gates NEW carrier switches, never re-confirming
    # the current carrier.
    velocity_check_window = 4
    min_speed_change_mps = 2.0
    min_angle_change_deg = 20.0
    fps = 25.0

    decision_sources = {'detected', 'smoothed'}
    display_sources = {'detected', 'smoothed'}

carrier_cfg = CarrierConfig()


## 11. Carrier assignment

In [11]:
import math

def player_bbox_distance(ball_xy_px, bbox, margin):
    x0, y0, x1, y1 = bbox
    cx, cy = ball_xy_px
    dx = max(x0 - cx, 0, cx - x1)
    dy = max(y0 - cy, 0, cy - y1)
    dist = (dx ** 2 + dy ** 2) ** 0.5
    return dist, dist <= margin

def player_feet_pitch(bbox, H):
    """Bottom-center of the player's bbox (feet), projected to pitch space via
    the same homography used for the ball -- keeps both in the same coordinate
    system so distances are directly comparable."""
    x0, y0, x1, y1 = bbox
    feet_px = ((x0 + x1) / 2, y1)
    return px_to_pitch(feet_px, H)

def best_with_margin(candidates, margin_ratio):
    """candidates: list of (track_id, dist), ascending distance = better.
    Returns (best_id, best_dist, ambiguous). Two nearly-tied candidates (e.g. a
    50/50 duel) are flagged ambiguous instead of arbitrarily picking the nearest."""
    if not candidates:
        return None, None, False
    candidates = sorted(candidates, key=lambda c: c[1])
    best_id, best_dist = candidates[0]
    if len(candidates) == 1:
        return best_id, best_dist, False
    second_dist = candidates[1][1]
    ambiguous = second_dist <= 0 or best_dist >= margin_ratio * second_dist
    return best_id, best_dist, ambiguous

def ball_velocity_change(ball_tracked_cache, frame_idx, window, fps, total_frames):
    """Compares the ball's pitch-space velocity just before vs. just after
    frame_idx (using the full offline cache, so this is allowed to look ahead).
    Returns (speed_before, speed_after, angle_change_deg), or None if there isn't
    enough position data nearby to evaluate (e.g. near a lost-track boundary)."""
    def get_pt(f):
        e = ball_tracked_cache.get(f)
        return e['xy_pitch'] if e is not None else None

    f_before = max(frame_idx - window, 0)
    f_after = min(frame_idx + window, total_frames - 1)
    p_before, p_now, p_after = get_pt(f_before), get_pt(frame_idx), get_pt(f_after)
    if p_before is None or p_now is None or p_after is None:
        return None

    dt_before = (frame_idx - f_before) / fps
    dt_after = (f_after - frame_idx) / fps
    if dt_before <= 0 or dt_after <= 0:
        return None

    v_before = ((p_now[0] - p_before[0]) / dt_before, (p_now[1] - p_before[1]) / dt_before)
    v_after = ((p_after[0] - p_now[0]) / dt_after, (p_after[1] - p_now[1]) / dt_after)

    speed_before = (v_before[0] ** 2 + v_before[1] ** 2) ** 0.5
    speed_after = (v_after[0] ** 2 + v_after[1] ** 2) ** 0.5

    ang_before = math.atan2(v_before[1], v_before[0])
    ang_after = math.atan2(v_after[1], v_after[0])
    angle_diff = abs(math.degrees(ang_after - ang_before))
    angle_diff = min(angle_diff, 360.0 - angle_diff)

    return speed_before, speed_after, angle_diff

def run_carrier_assigner(ball_tracked_cache, tracking_cache, locked_class_by_id,
                          homography_cache, carrier_cfg: CarrierConfig, total_frames):
    ball_carrier_cache = {}
    current_carrier = None
    candidate_streak = defaultdict(int)
    no_candidate_run = 0   # consecutive frames with no accepted candidate

    for frame_idx in range(total_frames):
        ball_entry = ball_tracked_cache.get(frame_idx)
        frame_tracks = tracking_cache.get(frame_idx, {}).get('tracks', [])
        player_tracks = [t for t in frame_tracks
                          if locked_class_by_id.get(t['track_id']) in ('player', 'goalkeeper')]

        if ball_entry is None or ball_entry['xy_px'] is None or not player_tracks:
            no_candidate_run += 1
            if no_candidate_run > carrier_cfg.no_candidate_grace_frames:
                current_carrier = None
                candidate_streak.clear()
            ball_carrier_cache[frame_idx] = {
                'track_id': current_carrier,   # still shows the held carrier during the grace window
                'method': None,
                'ball_source': ball_entry['source'] if ball_entry else None,
            }
            continue

        ball_source = ball_entry['source']
        drive_decision = ball_source in carrier_cfg.decision_sources
        can_display = ball_source in carrier_cfg.display_sources

        best_candidate = None
        best_method = None
        best_dist = float('inf')

        if drive_decision:
            # Phase 1 -- bbox overlap, with margin-based ambiguity check
            touching = []
            for t in player_tracks:
                dist, is_touch = player_bbox_distance(
                    ball_entry['xy_px'], t['bbox'], carrier_cfg.bbox_overlap_margin_px
                )
                if is_touch:
                    touching.append((t['track_id'], dist))

            p1_id, p1_dist, p1_ambiguous = best_with_margin(touching, carrier_cfg.candidate_margin_ratio)

            if p1_id is not None and not p1_ambiguous:
                best_candidate, best_method, best_dist = p1_id, 'overlap', p1_dist
            elif p1_ambiguous:
                best_method = 'contested'   # a fair duel -- don't guess, fall through to grace-hold

            # Phase 2 -- proximity in pitch space, only if phase 1 found nothing
            # (not even a contested duel -- a contested duel should not be
            # overridden by a further-away proximity match)
            if best_candidate is None and best_method != 'contested' and ball_entry['xy_pitch'] is not None:
                H = homography_cache[frame_idx] if frame_idx < len(homography_cache) else None
                proximity = []
                for t in player_tracks:
                    feet_pitch = player_feet_pitch(t['bbox'], H)
                    if feet_pitch is None:
                        continue
                    pdist = ((ball_entry['xy_pitch'][0] - feet_pitch[0]) ** 2 +
                             (ball_entry['xy_pitch'][1] - feet_pitch[1]) ** 2) ** 0.5
                    if pdist <= carrier_cfg.max_carrier_distance_m:
                        proximity.append((t['track_id'], pdist))

                p2_id, p2_dist, p2_ambiguous = best_with_margin(proximity, carrier_cfg.candidate_margin_ratio)
                if p2_id is not None and not p2_ambiguous:
                    best_candidate, best_method, best_dist = p2_id, 'proximity', p2_dist
                elif p2_ambiguous:
                    best_method = 'contested'

            if best_candidate is None:
                no_candidate_run += 1
                if carrier_cfg.clear_on_no_candidate and no_candidate_run > carrier_cfg.no_candidate_grace_frames:
                    current_carrier = None
                    candidate_streak.clear()
            else:
                no_candidate_run = 0

        # --- hysteresis + velocity-consistency gate (only advances on decision-eligible frames) ---
        # FIX -- decay every non-winning candidate's streak each frame so
        # min_frames_to_switch is enforced as truly CONSECUTIVE frames, as the config
        # comment promises. Previously candidate_streak was a defaultdict that only
        # ever incremented the winning candidate's own counter and never reset anyone
        # else's -- so a candidate could accumulate its 10 frames of "lead" scattered
        # arbitrarily far apart (interrupted by other candidates winning frames in
        # between) and still trigger a switch. That let noisy, non-consecutive
        # brushes-past-the-ball during a scramble eventually flip the carrier, which
        # defeats the point of hysteresis.
        if drive_decision and best_candidate is not None:
            for cid in list(candidate_streak.keys()):
                if cid != best_candidate:
                    candidate_streak[cid] = 0
            if best_candidate == current_carrier:
                candidate_streak[best_candidate] = 0
            else:
                # NEW -- require a touch-like velocity change before counting this
                # frame as evidence for a switch. A through-ball merely passing near
                # a player usually leaves the ball's trajectory unchanged; a real
                # touch usually changes its speed and/or direction.
                touch_check = ball_velocity_change(
                    ball_tracked_cache, frame_idx, carrier_cfg.velocity_check_window,
                    carrier_cfg.fps, total_frames
                )
                touch_like = True
                if touch_check is not None:
                    speed_before, speed_after, angle_delta = touch_check
                    speed_delta = abs(speed_after - speed_before)
                    touch_like = (speed_delta >= carrier_cfg.min_speed_change_mps or
                                  angle_delta >= carrier_cfg.min_angle_change_deg)

                if touch_like:
                    candidate_streak[best_candidate] += 1
                    if candidate_streak[best_candidate] >= carrier_cfg.min_frames_to_switch:
                        current_carrier = best_candidate
                        candidate_streak.clear()
                # else: inconclusive evidence -- don't increment (but don't reset
                # other candidates' streaks either; a real touch a couple frames
                # later within the same duel can still complete the switch)

        ball_carrier_cache[frame_idx] = {
            'track_id': current_carrier if can_display else None,
            'method': best_method,
            'ball_source': ball_source,
        }

    return ball_carrier_cache

ball_carrier_cache = run_carrier_assigner(
    ball_tracked_cache, tracking_cache, locked_class_by_id, homography_cache, carrier_cfg, TOTAL_FRAMES
)

n_assigned = sum(1 for v in ball_carrier_cache.values() if v['track_id'] is not None)
n_contested = sum(1 for v in ball_carrier_cache.values() if v['method'] == 'contested')
print(f"Frames with a carrier assigned: {n_assigned}/{TOTAL_FRAMES} ({100*n_assigned/TOTAL_FRAMES:.1f}%)")
print(f"Frames flagged contested (ambiguous duel, held previous carrier): {n_contested}/{TOTAL_FRAMES} ({100*n_contested/TOTAL_FRAMES:.1f}%)")


Frames with a carrier assigned: 33423/70326 (47.5%)
Frames flagged contested (ambiguous duel, held previous carrier): 811/70326 (1.2%)


## 12. Save ball_carrier_cache

In [12]:
os.makedirs(os.path.dirname(paths.BALL_CARRIER_CACHE_PATH), exist_ok=True)
with open(paths.BALL_CARRIER_CACHE_PATH, 'wb') as f:
    pickle.dump(ball_carrier_cache, f)
print(f"Saved to {paths.BALL_CARRIER_CACHE_PATH}")

Saved to barca_atletico_first_half/cache/barca_atletico_first_half_ball_carrier_cache.pkl


---
# Annotation -- ball (yellow circle) + carrier (green triangle)

Draws directly onto the source video and writes an annotated copy. `predicted`
ball frames and cleared-carrier frames simply draw nothing that frame -- no
special-casing needed, since the caches already encode trust upstream.

## 13. Annotate and save video

In [ ]:
import cv2

VIDEO_PATH = paths.VIDEO_PATH
OUTPUT_PATH = paths.OUTPUT_VIDEO_PATH
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

# Ball color depends on the source of the position, so you can see at a glance
# how much of the trail is real detections vs. filled-in.
#   'detected' -- green,  a real detection this frame
#   'smoothed' -- yellow, RTS-interpolated within a gap that closed on both sides
# 'predicted' no longer exists -- unconfirmed forward extrapolation (no closing
# detection yet) is now treated as 'lost' (xy_px=None) and simply isn't drawn.
BALL_COLOR_BY_SOURCE = {
    'detected': (0, 200, 0),       # green,  BGR
    'smoothed': (0, 220, 220),     # yellow, BGR
}
BALL_RADIUS = 8
BALL_THICKNESS = 2             # outline; use -1 for filled

CARRIER_COLOR = (255, 165, 0)  # orange, BGR -- distinct from all ball colors above
CARRIER_TRIANGLE_SIZE = 16     # half-width/height of the triangle
CARRIER_THICKNESS = -1         # filled


def draw_ball(frame, frame_idx, ball_tracked_cache):
    entry = ball_tracked_cache.get(frame_idx)
    if entry is None or entry['xy_px'] is None:
        return frame
    color = BALL_COLOR_BY_SOURCE.get(entry['source'])
    if color is None:   # 'lost' or any other non-drawable source
        return frame
    center = (int(entry['xy_px'][0]), int(entry['xy_px'][1]))
    cv2.circle(frame, center, BALL_RADIUS, color, BALL_THICKNESS, lineType=cv2.LINE_AA)
    return frame


def draw_carrier(frame, frame_idx, ball_carrier_cache, tracking_cache):
    carrier_entry = ball_carrier_cache.get(frame_idx)
    if carrier_entry is None or carrier_entry['track_id'] is None:
        return frame

    track_id = carrier_entry['track_id']
    frame_tracks = tracking_cache.get(frame_idx, {}).get('tracks', [])
    track = next((t for t in frame_tracks if t['track_id'] == track_id), None)
    if track is None:
        return frame

    x0, y0, x1, y1 = track['bbox']
    cx, top_y = int((x0 + x1) / 2), int(y0)   # centered above the player's head

    s = CARRIER_TRIANGLE_SIZE
    apex_y = top_y - 6
    pts = np.array([
        [cx, apex_y + s],
        [cx - s, apex_y],
        [cx + s, apex_y],
    ], dtype=np.int32)

    cv2.drawContours(frame, [pts], 0, CARRIER_COLOR, CARRIER_THICKNESS, lineType=cv2.LINE_AA)
    return frame


cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Could not open {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')   # swap to 'avc1' if you need H.264 and have the codec
writer = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))
if not writer.isOpened():
    cap.release()
    raise RuntimeError(f"Could not open VideoWriter for {OUTPUT_PATH}")

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = draw_ball(frame, frame_idx, ball_tracked_cache)
    frame = draw_carrier(frame, frame_idx, ball_carrier_cache, tracking_cache)

    writer.write(frame)

    if frame_idx % 250 == 0:
        print(f"  frame {frame_idx}/{n_frames}")
    frame_idx += 1

cap.release()
writer.release()
print(f"Saved annotated video to {OUTPUT_PATH} ({frame_idx} frames written)")
